# 第二阶段：Adult 数据集上的第一个 Baseline

这个 notebook 对应项目的第二阶段目标：

1. 选择第一个真实表格数据集
2. 读取数据并检查目标列、特征类型、缺失值
3. 完成一次最基础预处理
4. 跑通一个 baseline 模型
5. 记录 `accuracy` 和运行时间

这一阶段的重点不是追求最高分，而是建立你的第一条完整实验链路。


## 1. 为什么先选 Adult 数据集

`Adult` 是一个非常经典的表格分类数据集，很适合做项目的第一个正式实验。

原因有 4 个：

- 它是标准的分类任务，目标明确
- 既有数值特征，也有类别特征
- 还包含真实缺失值，能练到预处理
- 数据量适中，笔记本电脑也能跑

这个数据集的任务可以简单理解成：

> 根据一个人的人口统计信息和工作信息，预测其收入是否大于 50K。


In [ ]:
import time
from pathlib import Path

import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## 2. 读取数据

这里我们优先读取本地缓存；如果第一次运行还没有缓存，再从 `OpenML` 下载 `Adult` 数据集。

这样做有两个好处：

- 第二次运行会更快
- 网络波动时不容易卡住

说明：

- `data` 是特征表
- `target` 是目标列
- `as_frame=True` 让我们直接拿到 pandas DataFrame，便于检查列类型和缺失值


In [ ]:
project_root = Path.cwd()
if not (project_root / 'results').exists():
    project_root = project_root.parent

cache_path = project_root / 'data' / 'raw' / 'adult_openml.csv'
target_col = 'class'

if cache_path.exists():
    print(f'从本地缓存读取数据：{cache_path}')
    adult_df = pd.read_csv(cache_path)
    X = adult_df.drop(columns=[target_col])
    y = adult_df[target_col].copy()
else:
    print('首次运行会从 OpenML 下载 Adult 数据集，这一步如果网络慢，看起来会像卡住。')
    fetch_start = time.perf_counter()
    adult = fetch_openml(name='adult', version=2, as_frame=True)
    print(f'OpenML 下载完成，用时 {time.perf_counter() - fetch_start:.2f} 秒')
    X = adult.data.copy()
    y = adult.target.copy()
    adult_df = X.copy()
    adult_df[target_col] = y
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    adult_df.to_csv(cache_path, index=False)
    print(f'已写入本地缓存：{cache_path.resolve()}')

print('样本数 =', len(X))
print('原始特征数 =', X.shape[1])
print('目标列名称 =', y.name)
print('\n目标类别分布：')
print(y.value_counts())

X.head()


## 3. 先做数据体检：列类型、缺失值、任务特点

这是第二阶段最重要的学习点之一。

在真正训练模型之前，你应该先回答这些问题：

1. 哪些列是数值特征？
2. 哪些列是类别特征？
3. 哪些列有缺失值？缺多少？
4. 目标类别是否平衡？

这些判断会直接决定后面怎么做预处理。


In [ ]:
profile_df = pd.DataFrame({
    'dtype': X.dtypes.astype(str),
    'missing_count': X.isna().sum(),
})
profile_df['missing_rate'] = (profile_df['missing_count'] / len(X)).round(4)
profile_df = profile_df.sort_values(['missing_count', 'dtype'], ascending=[False, True])

print('总缺失值个数 =', int(X.isna().sum().sum()))
profile_df


从上面的输出里，你应该能观察到：

- `workclass`、`occupation`、`native-country` 有缺失值
- 一部分列是 `category` 类型，它们就是类别特征
- 目标分布有些不平衡，`<=50K` 明显更多

这意味着：

- 数值列和类别列最好分开处理
- 缺失值不能直接忽略
- 后面虽然先用 `accuracy`，但要知道它不是唯一指标


## 4. 划分训练集和测试集

我们继续沿用第一阶段学过的思路：

- 用训练集让模型学习
- 用测试集检查模型在新数据上的表现

这里继续使用：

- `test_size=0.2`
- `random_state=42`
- `stratify=y`

其中 `stratify=y` 仍然很重要，因为这是一个分类任务，而且类别并不完全平衡。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('训练集大小 =', len(X_train))
print('测试集大小 =', len(X_test))
print('\n训练集目标分布：')
print(y_train.value_counts())
print('\n测试集目标分布：')
print(y_test.value_counts())


## 5. 最基础预处理方案

这一步先不追求复杂，只做一个入门级但规范的处理：

- 数值特征：用中位数填补缺失值
- 类别特征：用最常见值填补缺失值，再做 one-hot 编码

为什么这样做？

- 中位数对极端值更稳一些
- 类别列不能直接给很多模型使用，所以需要编码
- `Pipeline + ColumnTransformer` 可以把流程写得清楚且可复用


In [ ]:
categorical_cols = X.select_dtypes(include=['category', 'object', 'str']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['category', 'object', 'str']).columns.tolist()

print('类别特征列：', categorical_cols)
print('数值特征列：', numeric_cols)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_cols,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            categorical_cols,
        ),
    ]
)

preprocessor.set_output(transform='pandas')


## 6. 训练第一个 Baseline：LightGBM

这里我们选 `LightGBM` 作为第一个 baseline，原因是：

- 它是表格任务里非常常见的强 baseline
- 在 CPU 上速度快
- 很适合作为你后面和 TabPFN / TabICL 比较的参照物

注意：

- 这一步我们还没有调参，只是先跑通第一条完整链路
- `fit_seconds` 会随着机器环境和并行设置略有波动，所以重新运行时训练时间和历史记录不完全一样是正常的


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        n_jobs=1,
        verbose=-1,
    )),
])

print('开始训练 LightGBM...')
start = time.perf_counter()
model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start
print(f'训练完成，用时 {fit_seconds:.2f} 秒')

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
encoded_feature_count = len(model.named_steps['preprocessor'].get_feature_names_out())

result_df = pd.DataFrame([
    {
        'dataset': 'adult',
        'model': 'LightGBM',
        'metric': 'accuracy',
        'accuracy': round(accuracy, 4),
        'fit_seconds': round(fit_seconds, 4),
        'n_samples': len(X),
        'train_size': len(X_train),
        'test_size': len(X_test),
        'n_features_raw': X.shape[1],
        'n_features_after_encoding': encoded_feature_count,
        'n_numeric_features': len(numeric_cols),
        'n_categorical_features': len(categorical_cols),
    }
])

result_df


## 7. 如何解读这个结果

如果你能顺利跑出结果，那么第二阶段的核心目标其实就已经达成了。

这张结果表至少说明了 3 件事：

1. 你已经完成了第一个真实数据集的正式实验
2. 你已经从“只会看论文”进入“能跑通完整实验流程”的阶段
3. 这条实验链路之后可以直接复用到 TabPFN v2 和 TabICL 上

这里的 `accuracy` 不是最终结论，而是你的第一个可比较基线。

后面你接入 TabPFN v2 时，核心问题就会变成：

> 在同样的数据集、同样的测试集上，TabPFN v2 和这个 LightGBM baseline 相比，谁更准、谁更快？


In [ ]:
project_root = Path.cwd()
if not (project_root / 'results').exists():
    project_root = project_root.parent

official_output_path = project_root / 'results' / 'first_result.csv'
rerun_output_path = project_root / 'results' / 'adult_lightgbm_rerun.csv'

if official_output_path.exists():
    result_df.to_csv(rerun_output_path, index=False)
    print('检测到官方 baseline 结果已存在，本次结果保存到：', rerun_output_path.resolve())
else:
    result_df.to_csv(official_output_path, index=False)
    print('结果已保存到：', official_output_path.resolve())


## 8. 这一阶段你应该记住什么

完成这个 notebook 后，请你至少能说清下面几件事：

- 为什么 `Adult` 适合作为第一个真实实验数据集
- 这个数据集里哪些列是类别特征，哪些有缺失值
- 为什么我们要先做预处理再训练 baseline
- `LightGBM` 在这个数据集上的第一个 baseline 结果是多少
- 为什么这个结果现在更像“起点”，而不是“终点”


## 9. 下一步

当这个 baseline 跑通以后，第三阶段就顺理成章了：

1. 接入 `TabPFN v2`
2. 在同一个数据集上进行第一次正式对比
3. 保持测试集一致
4. 把结果写进 `docs/experiment_log.md`

也就是说，这个 notebook 不是一个终点，而是你后面所有正式模型对比实验的模板。
